In [ ]:
from collections.abc import Callable, Iterable
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from jazz_graph.data.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender, RandomWalkRecommender
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.experiment import BSideExperiment
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run

checkpoint_path = None
# checkpoint_path = '/workspace/experiments/2026-03-26_02-01-03_gnn_simCLR_graph_parquet'  # This was good, but I think we need to retrain on new data.
if checkpoint_path is None:
    checkpoint_path = str(find_most_recent_run('/workspace/experiments', 'simCLR'))

with open(Path(checkpoint_path) / 'config.json', 'r') as f:
    run_config = json.loads(f.read())
    nodes_data_path = run_config['data_config'].get('dataset')

experiment_config = {
    'gnn': checkpoint_path,
    'nodes_data': nodes_data_path,
}


In [ ]:
run_config

In [ ]:
def lookup_from_dataset(data: HeteroData) -> LookupRecordings:
    # But why? Answer: the graph data may include pruning that is not part of the
    # parquet datasets or recording traits.
    rec_ids = data['performance'].x[:, 1].numpy()
    ids = np.arange(rec_ids.size)
    df = pd.DataFrame({'recording_ids': rec_ids, 'ids': ids}).set_index('recording_ids')
    return LookupRecordings(df)


In [ ]:
class UnsupervisedModelAdapter(torch.nn.Module):
    def __init__(self, model: UnsupervisedJazzModel):
        super().__init__()
        self.model = model

    def __call__(self, x_dict, edge_index_dict, batch):
        return self.model(batch)
        return self.model.encode(batch)

In [ ]:
def iter_dirs(root: str):
    import os
    for base_dir in os.listdir(root):
        print(base_dir)
        break

iter_dirs('/workspace/experiments')

In [ ]:
## Inductive Graph Recommender
from sklearn import model_selection
from jazz_graph.data.graph_builder import make_jazz_data
from jazz_graph.model.model import JazzModel
from jazz_graph.recommendation.recommender import InferenceRecommender


graph_data: HeteroData = make_jazz_data(CreateTensors(nodes_data_path))
model_state = load_model(checkpoint_path)
model_state = model_state.get('model_state_dict', model_state)
# model_state = model_state.get('model_state_dict', model_state)
model = UnsupervisedJazzModel.from_config(run_config)
model.load_state_dict(model_state)
model = UnsupervisedModelAdapter(model)
recommender = InferenceRecommender(model, graph_data)
lookup = recommender.lookup_recordings
recording_traits = fetch_recording_traits(use_proto=nodes_data_path.endswith('proto')).set_index('recording_id')
experimenter = BSideExperiment(recommender, recording_traits)

In [ ]:

album_experiments = [
    ('Miles Davis', 'Kind of Blue'),
    ('Miles Davis', 'Sketches of Spain'),
    ('Art Blakey & The Jazz Messengers', 'Mosaic'),
    ('Charles Mingus', "Mingus Ah Um"),  # lots of songs, should have some.
    ('The Dave Brubeck Quartet', "Time Out"),
    ('Ornette Coleman', 'The Shape of Jazz to Come')  # very unusual music--should probably be easy.
]

In [ ]:
recording_traits.dtypes

In [ ]:
experiment = album_experiments[-2]
print(experiment)
result = experimenter.b_side_precision(*experiment)
result
recording_traits.loc[result['recommended_ids']]

In [ ]:
results = experimenter.b_side_experiment(experiment_config, album_experiments)
pd.DataFrame.from_records(results).drop(columns=['recommended_ids'])

## Spotify Data

In [ ]:
import os
import json
from jazz_graph.recommendation.playlist import SpotifyListens
from jazz_graph.recommendation.recommender import ArtistWeightedRecommender
from jazz_graph.clean.data_normalization import normalize_title
import random

from jazz_graph.recommendation.experiment import SpotifyExperiement, RandomAlbumSplit


In [ ]:
spotify = 'local_data'
spotify_sample = '../local_data/my_spotify_data/spotify_extended_streaming_history/Streaming_History_Audio_2024-2025_4.json'
with open(spotify_sample, 'r') as f:
    spotify_data = json.loads(f.read())

def spotify_details(record: dict):
    details = [
        'ts', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name',
        'reason_start', 'reason_end', 'shuffle', 'skipped'
    ]
    return {key: record.get(key) for key in details}

spotify_data[-100:-90]
spotify_data[-10:]
ten_recent = [spotify_details(record) for record in spotify_data[-20:-10]]

In [ ]:

# seen = set()
# i = 0
# for rec in spotify_data:
#     title = rec['master_metadata_track_name']
#     norm_title = normalize_title(title)
#     if norm_title in seen:
#         continue
#     seen.add(norm_title)
#     if 'digital' in title.lower():
#         i += 1
#         print(norm_title)
#         print(title)
#     # print(norm_title)
# print(i)

In [ ]:
def test_rand_walk_recommender(graph_data):
    kob = recording_traits[recording_traits.album == "Kind of Blue"]
    baseline_recommender = RandomWalkRecommender(graph_data, 42)
    recommendations, scores, mask = baseline_recommender.get_recommendations(kob.index.to_list())
    ex_shape = recommendations.shape
    assert scores.shape == ex_shape
    assert mask.dtype == bool
    assert mask.shape == ex_shape
    return recommendations


# test_recs = test_rand_walk_recommender(graph_data)
# recording_traits.loc[test_recs]

In [ ]:
baseline_recommender = RandomWalkRecommender(graph_data, 42)
simple_artist_recommender = ArtistWeightedRecommender(recording_traits)


In [ ]:
# fields = 'master_metadata_track_name', 'master_metadata_album_album_name', 'master_metadata_album_artist_name'
# for recording in spotify_data:
#     for field in fields:
#         if recording.get(field) is None:
#             print(recording)

In [ ]:
experiment = SpotifyExperiement(recording_traits, spotify_data)

In [ ]:
experiment.run_experiment(baseline_recommender, {'model': "RandomWalkRecommender", "seed": 42})

In [ ]:
experiment.run_experiment(simple_artist_recommender, {'model': "ArtistWeightedRecommender"})

In [ ]:
experiment.run_experiment(recommender, experiment_config)


In [ ]:
def load_rec_run(path):
    path = Path(path)
    with open(path / 'metrics.jsonl') as f:
        return json.loads(f.read())

results = load_rec_run(str(find_most_recent_run('/workspace/experiments', 'spotify_recommendations')))
# results = load_rec_run('/workspace/experiments/2026-03-24_16-58-12_spotify_recommendations')
print(results['samples'].keys())
recording_traits.loc
samples = results['samples']
assert np.setdiff1d(samples['true_positive_in_familiar_recommendation'], samples['true_positive_in_familiar_recommendation']).size == 0

In [ ]:
tp_familiar = results['samples']['true_positive_in_familiar_recommendation']
recording_traits.loc[tp_familiar].value_counts('artist')

In [ ]:
results['samples'].keys()

In [ ]:
false_negatives = results['samples']['false_negative_in_novel_sample']
true_positives = results['samples']['true_positive_in_novel_recommendation']
recording_traits.loc[false_negatives].sort_values('release_date')
recording_traits.loc[true_positives][:50]
novel_recs = results['samples']['top_1302_novel_recommendations']
recording_traits.loc[novel_recs].head(50)
new_novel_recs = np.setdiff1d(novel_recs, true_positives)
recording_traits.loc[new_novel_recs][400:450]

## Explore misses in Spotify Matching

In [ ]:
import re
recording_traits[(recording_traits.artist.str.contains('Coltrane')) & recording_traits.album.str.contains('live', flags=re.IGNORECASE)].head(50)
None

In [ ]:
find_misses = SpotifyListens(recording_traits)
misses = [e[0] for e in find_misses.get_spotify_misses(spotify_data)]
print(len(misses), len(spotify_data))

In [ ]:
[key for key in find_misses.lookup if key[2].startswith('wayne s') and 'ad' in key[1]]

In [ ]:
recording_traits[recording_traits['album'].str.contains("Adam")]

In [ ]:
misses[0]

In [ ]:
unique = set()
clean_misses = []
for record in misses:
    keys = 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name'

    details = spotify_details(record)
    key = tuple(details[e] for e in keys)
    if key in unique:
        continue
    details.pop('ts')

    e = {key: details[other_key] for key, other_key in zip(['song', 'artist', 'album', ], keys)}
    clean_misses.append(e)
    unique.add(key)


In [ ]:
clean_misses

In [ ]:
[e for e in clean_misses if 'Gloria' in e['song']]

In [ ]:


def match_recording_traits(recording_traits, artist=None, album=None):
    def starts_with_lower(key, match):
        value = recording_traits[key]
        return value.str.contains(match, flags=re.IGNORECASE)

    if artist and album:
        return recording_traits[starts_with_lower('artist', artist) & starts_with_lower('album', album)]
    elif artist:
        return recording_traits[starts_with_lower('artist', artist)]
    elif album:
        return recording_traits[starts_with_lower('album', album)]


In [ ]:
re.search('^(Sunday )?at the', 'At the Village', re.IGNORECASE)

In [ ]:

result = match_recording_traits(recording_traits, 'Wayne Shorter', album='Adam')
#print(len(result))
result.sort_values('release_date').sort_values('title')#.loc[1662631]

In [ ]:
sunday = [spotify_details(v) for v in misses if v['master_metadata_album_album_name'].startswith('Sunday')]
normalize_title(sunday[0]['master_metadata_track_name']) == normalize_title(result.sort_values('release_date').loc[1662631].title)


In [ ]:
(sunday[0]['master_metadata_track_name'], result.sort_values('release_date').loc[1662631].title)

In [ ]:

result = match_recording_traits(recording_traits, album="N")
print(len(result))
result

In [ ]:
recording_traits.loc[experiment.in_samples].query("artist.str.contains('John')")

In [ ]:
recording_traits.loc[experiment.out_samples].query("artist.str.contains('John')")

In [ ]:
novel_recs = recording_traits.loc[recommendations[~mask]][:500]
all_recs = recording_traits.loc[recommendations][:500].assign(rank=np.arange(1, 501))
novel_recs = novel_recs.merge(all_recs['rank'], left_index=True, right_index=True)
repeat_idx = all_recs.index.difference(novel_recs.index)
repeat_recs = all_recs.loc[repeat_idx]
repeat_recs.head(30)

In [ ]:
from jazz_graph.etl.extract_discogs import InMemDiscogs, is_jazz_album

discogs = InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album)

In [ ]:
mismatches =[
    ('A Love Supreme', "song titles can be weird w/ Pt. I etc."),
    ('The Bill Evans Trio', "THE"),
    ('Equinox', 'Album naima, JC. Missing in discogs.'),
    ("Ascenseur pour l'échafaud", "This was  sound track by Miles in 1957. It's borked.")
    ("Feio", "no discogs match (Bitches Brew track.)")
]

In [ ]:
discogs.tracklist().get(normalize_title('naima'))

In [ ]:
discogs.get_albums_matching_title(normalize_title('a love supreme'))

In [ ]:
import jsonlines
with jsonlines.open(discogs.release_path) as f:
    for release in f:
        if normalize_title(release['title']) == normalize_title('the art of conversation'):
            print(release)

In [73]:
import psycopg
query = """
            SELECT DISTINCT  -- there are duplicates in recording_to_performer where a musican plays two instuments (Louis)
                recording_to_performer.artist_id,
                recording_to_performer.instrument,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
        """
query = """
            SELECT
                recording_to_performer.*,
                jazz_recordings.recording_id
            FROM
                jazz_recordings
            JOIN
                recording_to_performer ON jazz_recordings.recording_id = recording_to_performer.recording_id
            -- WHERE jazz_recordings.release_date >= %(start)s
            --     AND jazz_recordings.release_date < %(end)s
            -- LIMIT 10
        """
with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
    query_result = pd.read_sql(query, conn)
query_result

/tmp/ipykernel_43384/841734888.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_result = pd.read_sql(query, conn)


,artist_id,recording_id,artist_name,song_name,instrument,recording_id
0,384918,2061717,Garnett Brown,In All My Wildest Dreams,trombone,2061717
1,261871,8225410,Harry Carney,Do Nothin' Till You Hear From Me,saxophone,8225410
2,261871,8225410,Harry Carney,Do Nothin' Till You Hear From Me,clarinet,8225410
3,426811,27808228,Mickey Roker,Roses For Your Pillow,drums (drum set),27808228
4,612806,22853359,Urbie Green,It Need to Be Bee'd With,trombone,22853359
...,...,...,...,...,...,...
519618,1544115,31835957,Joachim Becker,Black Bottom Blues,keyboard,31835957
519619,892065,74021,John Oddo,You've Made Me So Very Happy,piano,74021
519620,892065,74612,John Oddo,Maria,keyboard,74612
519621,102999,563678,Art Taylor,Slowtrane,drums (drum set),563678


In [75]:
query_result[query_result.artist_name.str.startswith('Ella')]

,artist_id,recording_id,artist_name,song_name,instrument,recording_id
5152,1786,340361,Ella Fitzgerald,This Love That I’ve Found,lead vocals,340361
7711,2743598,36546230,Ella Russell,Love Song,choir vocals,36546230
9378,1786,472597,Ella Fitzgerald,Let’s Do It (Let’s Fall in Love) (alternate take),lead vocals,472597
13514,1786,1578580,Ella Fitzgerald,It’s Only a Paper Moon,lead vocals,1578580
14312,1786,1578530,Ella Fitzgerald,Hooray for Love,lead vocals,1578530
...,...,...,...,...,...,...
502918,1786,7468341,Ella Fitzgerald,But Not for Me (alternate take),lead vocals,7468341
511973,1786,2282969,Ella Fitzgerald,Lush Life,lead vocals,2282969
512767,1786,8829115,Ella Fitzgerald,You Go to My Head,guest,8829115
515171,1786,20916243,Ella Fitzgerald,This Love of Mine,lead vocals,20916243


In [ ]:
for row in query_result.instrument.value_counts():
    ...
for key, value in query_result.instrument.value_counts().items():
    print(key, value)

In [ ]:
query_result.album.loc[0]

In [ ]:
for row in query_result.itertuples():
    print(discogs.matching_discog(row))

In [ ]:
from jazz_graph.etl.extract_discogs import MatchDiscogs, InMemDiscogs, is_jazz_album
discogs = MatchDiscogs(InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album))

In [ ]:
discogs.matching_discog((None, None, "Adam's Apple", "Adam's Apple", "Wayne Shorter"))

In [ ]:
embeddings = torch.tensor([
    [.1, .1],
    [1, 0],
    [.1, .2],
    [.2, .1],
    [.3, .3]
])

dp = embeddings @ embeddings[[1, 3]].T
dp
dp.sum(dim=-1)

In [ ]:
INSTRUMENT_CATEGORIES = {
    # Drums
    "drums (drum set)": "drums",
    "electronic drum set": "drums (electronic)",
    "drum machine": "drums (electronic)",

    # Acoustic Piano
    "piano": "piano (acoustic)",
    "grand piano": "piano (acoustic)",
    "upright piano": "piano (acoustic)",
    "prepared piano": "piano (acoustic)",
    "fortepiano": "piano (acoustic)",
    "tack piano": "piano (acoustic)",
    "piano spinet": "piano (acoustic)",

    # Electric Piano / Keys (non-organ)
    "electric piano": "piano (electric)",
    "Rhodes piano": "piano (electric)",
    "Wurlitzer electric piano": "piano (electric)",
    "electric grand piano": "piano (electric)",
    "clavinet": "piano (electric)",
    "celesta": "piano (electric)",
    "harpsichord": "piano (electric)",
    "synthesizer": "piano (electric)",
    "analog synthesizer": "piano (electric)",
    "bass synthesizer": "piano (electric)",
    "string synthesizer": "piano (electric)",
    "keyboard": "piano (electric)",
    "mellotron": "piano (electric)",
    "Moog": "piano (electric)",
    "Minimoog": "piano (electric)",
    "synclavier": "piano (electric)",

    # Organ (distinct jazz tradition)
    "organ": "organ",
    "Hammond organ": "organ",
    "electronic organ": "organ",
    "pipe organ": "organ",
    "theatre organ": "organ",
    "harmonium": "organ",
    "barrel organ": "organ",

    # Double Bass (acoustic)
    "double bass": "bass (acoustic)",
    "acoustic bass guitar": "bass (acoustic)",
    "bass violin": "bass (acoustic)",
    "bass viol": "bass (acoustic)",
    "contrabass saxophone": "bass (acoustic)",  # functional bass role

    # Electric Bass
    "electric bass guitar": "bass (electric)",
    "bass guitar": "bass (electric)",
    "fretless bass": "bass (electric)",
    "electric bass guitar": "bass (electric)",
    "electric upright bass": "bass (electric)",
    "bass pedals": "bass (electric)",
    "keyboard bass": "bass (electric)",
    "washtub bass": "bass (electric)",

    # Acoustic Guitar
    "guitar": "guitar (acoustic)",
    "acoustic guitar": "guitar (acoustic)",
    "classical guitar": "guitar (acoustic)",
    "steel-string acoustic guitar": "guitar (acoustic)",
    "archtop guitar": "guitar (acoustic)",
    "resonator guitar": "guitar (acoustic)",
    "12 string guitar": "guitar (acoustic)",
    "acoustic fretless guitar": "guitar (acoustic)",

    # Electric Guitar
    "electric guitar": "guitar (electric)",
    "guitar synthesizer": "guitar (electric)",
    "pedal steel guitar": "guitar (electric)",
    "lap steel guitar": "guitar (electric)",
    "steel guitar": "guitar (electric)",
    "baritone guitar": "guitar (electric)",
    "electric fretless guitar": "guitar (electric)",
    "ebow": "guitar (electric)",
    "Chapman stick": "guitar (electric)",
    "slide guitar": "guitar (electric)",

    # Trumpet family (keep together — flugelhorn/cornet are same acoustic tradition)
    "trumpet": "trumpet",
    "flugelhorn": "trumpet",
    "cornet": "trumpet",
    "pocket trumpet": "trumpet",
    "piccolo trumpet": "trumpet",
    "bass trumpet": "trumpet",
    "bugle": "trumpet",

    # Trombone
    "trombone": "trombone",
    "bass trombone": "trombone",
    "valve trombone": "trombone",
    "tenor trombone": "trombone",

    # Saxophones — keep all voices distinct
    "soprano saxophone": "saxophone (soprano)",
    "sopranino saxophone": "saxophone (soprano)",
    "alto saxophone": "saxophone (alto)",
    "tenor saxophone": "saxophone (tenor)",
    "baritone saxophone": "saxophone (baritone)",
    "bass saxophone": "saxophone (baritone)",  # close enough
    "saxophone": "saxophone (tenor)",  # generic, tenor is the default assumption in jazz

    # Clarinet
    "clarinet": "clarinet",
    "bass clarinet": "clarinet",
    "alto clarinet": "clarinet",
    "contrabass clarinet": "clarinet",
    "basset clarinet": "clarinet",

    # Flute
    "flute": "flute",
    "alto flute": "flute",
    "bass flute": "flute",
    "piccolo": "flute",
    "recorder": "flute",

    # Vibraphone (signature jazz melodic percussion)
    "vibraphone": "vibraphone",
    "marimba": "vibraphone",  # close enough functionally

    # Violin
    "violin": "violin",
    "electric violin": "violin",
    "fiddle": "violin",
    "viola": "violin",
    "cello": "violin",
    "electric cello": "violin",
    "string quartet": "violin",
    "strings": "violin",
    "bass violin": "violin",
    "tenor violin": "violin",

    # Vocals
    "lead vocals": "vocals (lead)",
    "background vocals": "vocals (other)",
    "choir vocals": "vocals (other)",
    "spoken vocals": "vocals (other)",
    "tenor vocals": "vocals (other)",
    "soprano vocals": "vocals (other)",
    "baritone vocals": "vocals (other)",
    "alto vocals": "vocals (other)",
    "bass vocals": "vocals (other)",
    "vocal": "vocals (other)",
    "other vocals": "vocals (other)",
    "mezzo-soprano vocals": "vocals (other)",
    "countertenor vocals": "vocals (other)",
    "whistling": "vocals (other)",

    # Other brass
    "French horn": "other brass",
    "tuba": "other brass",
    "sousaphone": "other brass",
    "mellophone": "other brass",
    "euphonium": "other brass",
    "baritone horn": "other brass",
    "horn": "other brass",
    "tenor horn / alto horn": "other brass",
    "brass": "other brass",
    "valved brass instruments": "other brass",
    "cornett": "other brass",
    "sackbut": "other brass",

    # Other wind
    "bassoon": "other wind",
    "contrabassoon": "other wind",
    "oboe": "other wind",
    "cor anglais": "other wind",
    "woodwind": "other wind",
    "reeds": "other wind",
    "wind instruments": "other wind",
    "double reed": "other wind",
    "harmonica": "other wind",
    "melodica": "other wind",
    "accordion": "other wind",
    "bandoneón": "other wind",
    "bagpipe": "other wind",
    "concertina": "other wind",

    # Percussion (not drum set)
    "percussion": "percussion",
    "membranophone": "percussion",
    "congas": "percussion",
    "bongos": "percussion",
    "timbales": "percussion",
    "snare drum": "percussion",
    "bass drum": "percussion",
    "tambourine": "percussion",
    "cowbell": "percussion",
    "djembe": "percussion",
    "tabla": "percussion",
    "shekere": "percussion",
    "talking drum": "percussion",
    "gong": "percussion",
    "maracas": "percussion",
    "triangle": "percussion",
    "timpani": "percussion",
    "tubular bells": "percussion",
    "xylophone": "percussion",
    "glockenspiel": "percussion",
    "bell": "percussion",
    "cymbal": "percussion",
    "chimes": "percussion",
    "handclaps": "percussion",
    "shakers": "percussion",
    "cimbalom": "percussion",
    "vibraslap": "percussion",
    "finger snaps": "percussion",
    "foot stomps": "percussion",
    "castanets": "percussion",
    "washboard": "percussion",
    "steelpan": "percussion",
    "marimba": "percussion",
    "finger cymbals": "percussion",
    "hi-hat": "percussion",
    "tom-tom": "percussion",
    "frame drum": "percussion",
    "slit drum": "percussion",
    "Batá drum": "percussion",
    "dunun": "percussion",
    "bendir": "percussion",
    "surdo": "percussion",
    "timbales": "percussion",
    "claves": "percussion",
    "cabasa": "percussion",
    "bongos": "percussion",
    "cuíca": "percussion",
    "berimbau": "percussion",
    "wood block": "percussion",
    "gankogui": "percussion",
    "agogô": "percussion",
    "handbell": "percussion",
    "crotales": "percussion",
    "ganzá": "percussion",
    "ocean drum": "percussion",
    "rainstick": "percussion",
    "rhythm sticks": "percussion",
    "bones": "percussion",
    "darbuka": "percussion",
    "goblet drum": "percussion",
    "sabar": "percussion",
    "zabumba": "percussion",
    "ashiko": "percussion",
    "quinto": "percussion",
    "dohol": "percussion",
    "junjung": "percussion",
    "water drum": "percussion",
    "friction drum": "percussion",
    "mridangam": "percussion",
    "kanjira": "percussion",
    "ghatam": "percussion",

    # Electronic / effects
    "effects": "electronic/effects",
    "sampler": "electronic/effects",
    "vocoder": "electronic/effects",
    "turntable": "electronic/effects",
    "tape": "electronic/effects",
    "EWI": "electronic/effects",
    "Lyricon": "electronic/effects",
    "wind synthesizer": "electronic/effects",
    "talkbox": "electronic/effects",
    "voice synthesizer": "electronic/effects",
    "theremin": "electronic/effects",

    # World / other instruments (too sparse or too niche to signal anything)
    "banjo": "other/world",
    "tenor banjo": "other/world",
    "mandolin": "other/world",
    "ukulele": "other/world",
    "harp": "other/world",
    "electric harp": "other/world",
    "harp guitar": "other/world",
    "oud": "other/world",
    "sitar": "other/world",
    "koto": "other/world",
    "lute": "other/world",
    "zither": "other/world",
    "kora": "other/world",
    "bouzouki": "other/world",
    "charango": "other/world",
    "banjo": "other/world",
    "mbira": "other/world",
    "shakuhachi": "other/world",
    "didgeridoo": "other/world",
    "kazoo": "other/world",
    "whistle": "other/world",
    "tin whistle": "other/world",
    "autoharp": "other/world",
    "guitar family": "other/world",
    "violin family": "other/world",
    "trumpet family": "other/world",
    "bowed string instruments": "other/world",
    "slide brass instruments": "other/world",
    "lamellaphone": "other/world",
    "idiophone": "other/world",
    "shruti box": "other/world",
    "tambura": "other/world",
    "harmonium": "other/world",
    "accordion": "other/world",
    "mandola": "other/world",
    "mandocello": "other/world",
    "cavaquinho": "other/world",
    "fiddle": "other/world",
    "rebab": "other/world",
    "hardingfele": "other/world",
    "bandura": "other/world",
    "Chapman stick": "other/world",
    "guzheng": "other/world",
    "pipa": "other/world",
    "khamak": "other/world",
    "sarod": "other/world",
    "santoor": "other/world",
    "kamancheh": "other/world",
    "setar": "other/world",
    "tar": "other/world",
    "sitar": "other/world",
    "tabla": "other/world",
    "tanpura": "other/world",
    "valiha": "other/world",
    "ngɔni": "other/world",
    "xalam": "other/world",
    "gumbri": "other/world",
    "balafon": "other/world",
    "musical saw": "other/world",
    "waterphone": "other/world",
    "singing bowl": "other/world",
    "glass harp": "other/world",
    "toy piano": "other/world",
    "clavichord": "other/world",
    "harpsichord": "other/world",
    "psaltery": "other/world",
    "dulcimer": "other/world",
    "hammered dulcimer": "other/world",
    "omnichord": "other/world",
    "marímbula": "other/world",
    "taragot": "other/world",
    "duduk": "other/world",
    "suona": "other/world",
    "shehnai": "other/world",
    "kaval": "other/world",
    "ney": "other/world",
    "bansuri": "other/world",
    "quena": "other/world",
    "pan flute": "other/world",
    "shakuhachi": "other/world",
    "transverse flute": "other/world",
    "alto flute": "other/world",
    "sheng": "other/world",
    "fife": "other/world",
    "farfisa": "other/world",
    "tubax": "other/world",
    "tiple": "other/world",
    "charango": "other/world",
    "cuatro": "other/world",
    "berimbau": "other/world",
    "udu": "other/world",
    "caxixi": "other/world",
    "güiro": "other/world",
    "shofar": "other/world",

    # Other personnel (not instruments)
    "guest": "other personnel",
    "assistant": "other personnel",
    "executive": "other personnel",
    "associate": "other personnel",
    "additional": "other personnel",
    "co": "other personnel",
    "task": "other personnel",
    "solo": "other personnel",
    "other instruments": "other personnel",
}

pd.Series(INSTRUMENT_CATEGORIES).value_counts()

other/world             90
percussion              67
piano (electric)        15
vocals                  14
other wind              13
electronic/effects      12
other brass             12
violin                   9
guitar (electric)        9
other personnel          9
guitar (acoustic)        8
piano (acoustic)         7
bass (electric)          7
trumpet                  7
organ                    6
clarinet                 5
bass (acoustic)          4
trombone                 4
flute                    4
drums                    2
saxophone (soprano)      2
saxophone (tenor)        2
saxophone (baritone)     2
saxophone (alto)         1
vibraphone               1
Name: count, dtype: int64

In [ ]:
from torch_geometric.nn import GATv2Conv, GATConv, SAGEConv

x_dict = torch.tensor([
    [.1, .2, .3, .4],
    [.1, -.3, .5, .6]
])
edge_index_dict = torch.tensor([
    [1, 0], [0, 1]
])
edge_attrs = torch.tensor([
    [-1., .1, -.8, .2],
    [-1., .1, .8, .2]
])

conv = GATv2Conv(4, 6, edge_dim=4)
conv(x_dict, edge_index_dict, edge_attrs)

tensor([[-0.5755,  0.2392,  0.0149, -0.3725,  0.1440,  0.0401],
        [-0.5768,  0.2399,  0.0138, -0.3711,  0.1454,  0.0400]],
       grad_fn=<AddBackward0>)

In [ ]:
from torch import nn
from torch_geometric.nn import HeteroConv


class GNNModel(nn.Module):
    def __init__(self, hidden_dim, input_dim, metadata, edge_dim: dict[tuple[str, str, str], int], model_type: str | Sequence[str] = 'gatv2', num_layers=3, dropout=0.):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)

        # Create layers dynamically
        self.convs = nn.ModuleList()
        edge_types = metadata[1]
        if isinstance(model_type, str):
            model_type = [model_type for _ in range(len(edge_types))]
        else:
            model_type = list(model_type)
        # assert isinstance(model_type, list)
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            self.convs.append(
                HeteroConv({
                    key: self._model_type(model_type[i], in_dim, hidden_dim, edge_dim.get(key)) for key in edge_types
                })
            )

    def _model_type(self, model_type, in_dim, hidden_dim, edge_dim):
        if model_type == 'sage':
            if edge_dim is not None:
                raise ValueError("SAGEConv is not supported with edge attributes.")
            return SAGEConv(in_dim, hidden_dim, normalize=True)
        elif model_type == 'gat':
            return GATConv(in_dim, hidden_dim, edge_dim=edge_dim, add_self_loops=False)
        elif model_type == 'gatv2':
            return GATv2Conv(in_dim, hidden_dim, edge_dim=edge_dim, add_self_loops=False)
        else:
            raise ValueError("Expected model type 'sage' or 'gat'.")

    def forward(self, x_dict, edge_dict, edge_attrs):
        for i, conv in enumerate(self.convs):
            x_dict = conv(x_dict, edge_dict, edge_attrs)

            # Apply ReLU + dropout to all layers except the last
            if i < self.num_layers - 1:
                x_dict = {key: F.normalize(self.dropout(F.relu(val)), dim=-1) for key, val in x_dict.items()}
            else:
                # Last layer: dropout only (no ReLU)
                x_dict = {key: F.normalize(self.dropout(val), dim=-1) for key, val in x_dict.items()}

        return x_dict